In [1]:
import sys
import json
# TO CHANGE
BASEDIR = "../../.."
sys.path.insert(0, BASEDIR)

## Пример инициализации PersonalAI-класса

In [2]:
from pprint import pprint
import joblib

In [3]:
from src import PersonalAI, PersonalAIConfig
from src.kg_model import KnowledgeGraphModelConfig
from src.db_drivers.vector_driver.embedders import EmbedderModelConfig

from src.pipelines.qa import QAPipelineConfig
from src.pipelines.qa.kg_reasoning import KnowledgeGraphReasonerConfig
from src.pipelines.qa.query_preprocessing import QueryPreprocessorConfig
from src.pipelines.qa.query_preprocessing.decomposition import QueryDecomposerConfig
from src.pipelines.qa.query_preprocessing.denoising import QueryDenoiserConfig
from src.pipelines.qa.query_preprocessing.enhancing import QueryEnhancerConfig
from src.pipelines.qa.kg_reasoning.weak_reasoner import WeakKGReasonerConfig
from src.pipelines.qa.kg_reasoning.medium_reasoner import MediumKGReasonerConfig
from src.pipelines.qa.answers_aggregation import AnswersAggregatorConfig

from src.pipelines.qa.knowledge_retriever import KnowledgeRetrieverConfig
from src.pipelines.qa.knowledge_retriever.traversal_methods import WaterCirclesSearchConfig

from src.db_drivers.kv_driver.configs import DEFAULT_MIXEDKV_CONFIG
from src.db_drivers.kv_driver import KeyValueDriverConfig
from src.utils.logger import LogLevel, Logger
from src.utils import ModuleType

/home/dzigen/miniconda3/envs/pai/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
kg_config = KnowledgeGraphModelConfig(
    nodestree_config=None # Модель дерева вершин строиться не будет
)
EMBEDDER_MODEL_PATH = '../../../models/intfloat/multilingual-e5-base' # PATH TO APPROPRIATE EMBEDDER-MODEL
kg_config.embedders_configs['m-e5-base'] = EmbedderModelConfig(model_name_or_path=EMBEDDER_MODEL_PATH)

In [5]:
pai_config = PersonalAIConfig(
    lang='ru',
    log_level=LogLevel.DEBUG,
    kg_model_config=kg_config,
    qa_pipeline_config=QAPipelineConfig(
        preprocessor_config=QueryPreprocessorConfig(
            denoising_config=QueryDenoiserConfig(),
            enhancing_config=None,
            decomposition_config=QueryDecomposerConfig()
        ),
        reasoner_config=KnowledgeGraphReasonerConfig(
            reasoner_name='medium', # 'weak' | 'medium'
            reasoner_config=MediumKGReasonerConfig(
                enable_searchplan_steps_check=False,
                planlimit_exceeded_behaviour='casual_answer'
            )  # WeakKGReasonerConfig() |  MediumKGReasonerConfig()
        ),
        aggregator_config=AnswersAggregatorConfig(
            answers_summ_behaviour='casual_answer'
        )
    )
)

In [6]:
personalai = PersonalAI(pai_config)

New PAI environment will be created.


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1390.92it/s]


In [7]:
personalai.open_connections()

In [8]:
personalai.mem_pipeline.clear_kv_caches()
personalai.qa_pipeline.clear_kv_caches(
    #clear_traversal_cache=True,
    #clear_retrieval_cache=True
)

In [9]:
personalai.mem_pipeline.clear_agent_tgen_stat()
personalai.qa_pipeline.clear_agent_tgen_stat()

In [11]:
print("mem kv_cache info:")
print(json.dumps(personalai.mem_pipeline.get_cache_stat(), indent=5))
print("mem agent tgen info:")
print(json.dumps(personalai.mem_pipeline.get_agent_tgen_stat(), indent=5))

mem kv_cache info:
{
     "MemPipeline": null,
     "extractor": {
          "LLMExtractor": null,
          "triplets_extraction_solver": 0,
          "thesises_extraction_solver": 0
     },
     "updator": {
          "LLMUpdator": null,
          "replace_simple_solver": 0,
          "replace_hyper_solver": 0
     }
}
mem agent tgen info:
{
     "extractor": {
          "triplets_extraction_solver": {
               "prompt_tokens_amount": {
                    "count": 0,
                    "count_not_null": 0,
                    "min": null,
                    "max": null,
                    "median": null,
                    "mean": null,
                    "std": null,
                    "sum": null
               },
               "generated_tokens_amount": {
                    "count": 0,
                    "count_not_null": 0,
                    "min": null,
                    "max": null,
                    "median": null,
                    "mean": null,
      

In [12]:
print("qa kv_cache info:")
print(json.dumps(personalai.qa_pipeline.get_cache_stat(), indent=5))
print("qa agent tgen info:")
print(json.dumps(personalai.qa_pipeline.get_agent_tgen_stat(), indent=5))

qa kv_cache info:
{
     "QAPipeline": 0,
     "query_preprocessor": {
          "QueryPreprocessor": 0,
          "denoiser": {
               "QueryDenoiser": 0,
               "swremoval_solver": 0,
               "grammar_check_solver": 0
          },
          "enhancer": null,
          "decomposer": {
               "QueryDecomposer": 0,
               "decompose_classifier_solver": 0,
               "q_decomposition_solver": 0
          }
     },
     "kg_reasoner": {
          "KnowledgeGraphReasoner": null,
          "reasoner": {
               "MediumKGReasoner": 0,
               "searchplan_enhancer": {
                    "SearchPlanEnhancer": 0,
                    "plan_initialing_solver": 0,
                    "enhance_classify_solver": 0,
                    "plan_enhancing_solver": 0,
                    "searchstop_classify_solver": 0
               },
               "entities_extractor": {
                    "EntitiesExtractor": 0,
                    "entities_

In [13]:
personalai.kg_model.clear()
personalai.textid_store.clear_store()

In [14]:
print(json.dumps(personalai.kg_model.count_items(detailed=True),indent=5))

{
     "graph_info": {
          "triplets": {
               "simple": 0,
               "hyper": 0,
               "episodic": 0,
               "time": 0
          },
          "nodes": {
               "object": 0,
               "hyper": 0,
               "episodic": 0,
               "time": 0
          }
     },
     "embeddings_info": {
          "nodes": {
               "object": {
                    "dense_nodes": 0,
                    "bm25_nodes": 0
               },
               "hyper": {
                    "dense_nodes": 0,
                    "bm25_nodes": 0
               },
               "episodic": {
                    "dense_nodes": 0,
                    "bm25_nodes": 0
               },
               "time": {
                    "dense_nodes": 0,
                    "bm25_nodes": 0
               }
          },
          "triplets": {
               "dense_triplets": 0,
               "bm25_triplets": 0
          }
     },
     "nodestree_info": null
}


In [15]:
personalai.textid_store.count_items()

{'textid_to_tripletsid': 0, 'tripletid_to_textsid': 0}

In [16]:
TEXT_EXAMPLES = [
    "Проживающие в общежитии студенты имеют право rруглосуточного доступа к месту проживания.",
    "Проживающие в общежитии студенты имеют право обратиться к администрации ФГБУ «МСГ» с заявлением, заверенным заведующим общежития, о размещении в гостевых комнатах общежития родственников (на короткий период пребывания, не менее 2-х суток), родителей - на любой срок (при предоставлении документа, подтверждающего степень родства).",
    "Проживающие в общежитии студенты имеют право пользоваться помещениями для самостоятельных занятий и помещениями культурно-бытового назначения, оборудованием, инвентарем общежития.",
    "Проживающие в общежитии студенты имеют право обращаться к администрации корпуса с просьбами о своевременном ремонте, замене оборудования и инвентаря, вышедшего из строя не по их вине.",
    "Проживающие в общежитии студенты имеют право на переселение из одного помещения в другое в том же корпусе, а также на переселение из одного корпуса в другой при наличии свободных мест, с согласия администрации корпуса.",
    "Проживающие в общежитии студенты имеют право участвовать в формировании и выборах Студенческого совета МСГ и быть избранным в его состав.",
    "Проживающие в общежитии студенты имеют право участвовать (вносить предложения) через Студенческий совет МСГ и отдел по молодежной политике ФГБУ «МСГ» в решении вопросов совершенствования жилищно-бытовых условий, организации воспитательной работы и досуга.",
    "Проживающие в общежитии студенты имеют право принимать участие в общественных, спортивных и культурно-досуговых мероприятиях, организованных администрацией ФГБУ «МСГ» и Студенческим советом МСГ.",
    "Проживающие в общежитии студенты имеют право пользоваться разрешенной бытовой техникой с соблюдением правил техники безопасности и правил пожарной безопасности.",
    "Проживающие в общежитии студенты имеют право бесплатно посещать Межвузовский  учебно-спортивный центр в утвержденное администрацией ФГБУ «МСГ» и согласованное со Студенческим советом МСГ время.",
    "Проживающие в общежитии студенты имеют право бесплатно посещать душевой комплекс с сауной ФГБУ «МСГ» в отведенное администрацией ФГБУ «МСГ» время.",

    "Проживающие в общежитии студенты обязаны ознакомиться с настоящими Правилами.",
    "Проживающие в общежитии студенты обязаны cтрого соблюдать настоящие Правила, правила техники безопасности, правила пожарной безопасности, требования (рекомендации) Рсопотребнадзора.",
    "Проживающие в общежитии студенты обязаны принимать участие в проведении инструкторско-методических занятий по теме: «Организация эвакуации людей из студенческого общежития при возникновении пожара и чрезвычайных ситуациях».",
    "Проживающие в общежитии студенты обязаны при чрезвычайных ситуациях и срабатывании оповещения — покинуть общежитие.",
    "Проживающие в общежитии студенты обязаны выполнять условия договора найма жилого помещения.",
    "Проживающие в общежитии студенты обязаны В установленном порядке и сроки представлять документы для регистрации по месту пребывания, своевременно предоставлять в отдел учета, размещения и регистрации информацию о смене паспортных данных (20 лет — замена паспорта, смена фамилии и др. личные данные).",
    "Проживающие в общежитии студенты обязаны ежегодно представлять администрации корпуса копию справки о прохождении флюорографии (ФЛГ).",
    "Проживающие в общежитии студенты обязаны своевременно в соответствии с условиями договора найма жилого помещения в общежитии вносить плату в установленных размерах за проживание в общежитии и за все виды предоставляемых дополнительных платных услуг.",
    "Проживающие в общежитии студенты обязаны по требованию администрации корпуса предъявлять электронный пропуск установленного образца на право входа в общежитие.",

    "Проживающие в общежитии студенты обязаны обеспечить возможность осмотра жилой комнаты администрацией корпуса в присутствии членов Студенческого совета МСГ или проживающих на этаже студентов с целью контроля санитарного состояния, проверки сохранности имущества, проведения профилактических и других видов работ.",
    "Проживающие в общежитии студенты обязаны не допускать, а в случае обнаружения сообщать администрации корпуса и администрации ФГБУ «МСГ», любые попытки распространения информации или пропаганды сомнительных организаций (террористических), так как это противоречит действующему законодательству РФ.",
    "Проживающие в общежитии студенты обязаны принимать посетителей только в установленное администрацией ФГБУ «МСГ» время, согласно графику посещения гостей, утвержденному директором ФГБУ «МСГ». Время визитов для посторонних посетителей: среда с 17.00 до 21.00, суббота, воскресенье, праздничные дни с 13.00 до 21.00, время визитов для посетителей из ФГБУ «МСГ»: с понедельника по пятницу с 17.00 до 21.00, в субботу, воскресенье, праздничные дни с 13.00 до 21.00.",
    "Проживающие в общежитии студенты обязаны при временном отсутствии более чем на 3 суток (каникулы, практика, стажировка и др.) письменно уведомить администрацию корпуса не менее чем за 2 (два) дня до предполагаемого выбытия.",
    "Проживающие в общежитии студенты обязаныво время пользования помещениями для самостоятельных занятий и культурно-бытового назначения соблюдать тишину и не создавать препятствий другим проживающим в пользовании указанными помещениями.",
    "Проживающие в общежитии студенты обязаныбережно относиться к помещениям, оборудованию и инвентарю.",
    "Проживающие в общежитии студенты обязанынаходясь на кухне, проявлять повьшиенное внимание к работающим бытовым приборам (особенно к газовым плитам) во избежание задымления, возгорания (выключать кипящие чайники, кастрюли, подгорающую пищу и т.п.).",
    "Проживающие в общежитии студенты обязанына основании графика, установленного Старостой этажа, добросовестно нести обязанности по дежурству на кухне.",

    "Проживающие в общежитии студенты обязаныэкономно расходовать электроэнергию, газ и воду.",
    "Проживающие в общежитии студенты обязанысоблюдать чистоту и порядок в жилых помещениях и местах общего пользования, осуществлять влажную уборку жилого помещения не реже 3-х раз в неделю, в соответствии с графиком уборки комнаты.",
    "Проживающие в общежитии студенты обязаныв случае обнаружения неисправностей датчиков пожарной безопасности, электричества, отопительных батарей, инвентаря, оборудования и насекомых сообщить администрации корпуса либо заполнить форму заявки в системе АСУ «Заявки» по адресу: ВИр://арр.т1зе-зрь.га.",
    "Проживающие в общежитии студенты обязаны возмещать причиненный материальный ущерб в соответствии с действующим законодательством и договором найма жилого помещения.",
    "Проживающие в общежитии студенты обязаны соблюдать требования морально-этических норм поведения, поддерживать атмосферу доброжелательности и взаимного уважения по отношению к проживающим, работникам, администрации корпуса, администрации ФГБУ «МСГ» и членам Студенческого совета МСГ."
]
print(len(TEXT_EXAMPLES))

33


In [17]:
extracted_triplets = []
for i, example in enumerate(TEXT_EXAMPLES):
    print(f"{i+1}. {example}")
    update_results, _, _ = personalai.update_memory(example)
    tmp_text_id, tmp_extracted_triplets = update_results
    extracted_triplets += tmp_extracted_triplets
    print("text_id: ", tmp_text_id)
    print("extracted triplets: ", len(tmp_extracted_triplets))

1. Проживающие в общежитии студенты имеют право rруглосуточного доступа к месту проживания.
text_id:  b2a6b1356bbcd3a34c956956c1ca0ffa
extracted triplets:  17
2. Проживающие в общежитии студенты имеют право обратиться к администрации ФГБУ «МСГ» с заявлением, заверенным заведующим общежития, о размещении в гостевых комнатах общежития родственников (на короткий период пребывания, не менее 2-х суток), родителей - на любой срок (при предоставлении документа, подтверждающего степень родства).
text_id:  477e93013ea1a3a6f8e3a4c74564a167
extracted triplets:  48
3. Проживающие в общежитии студенты имеют право пользоваться помещениями для самостоятельных занятий и помещениями культурно-бытового назначения, оборудованием, инвентарем общежития.
text_id:  9e7efd24ea148ea48c2cc20e512d15d0
extracted triplets:  30
4. Проживающие в общежитии студенты имеют право обращаться к администрации корпуса с просьбами о своевременном ремонте, замене оборудования и инвентаря, вышедшего из строя не по их вине.
tex

In [18]:
print(json.dumps(personalai.kg_model.count_items(detailed=True),indent=5))
personalai.kg_model.check_consistency()

{
     "graph_info": {
          "triplets": {
               "simple": 165,
               "hyper": 411,
               "episodic": 602,
               "time": 0
          },
          "nodes": {
               "object": 352,
               "hyper": 125,
               "episodic": 33,
               "time": 0
          }
     },
     "embeddings_info": {
          "nodes": {
               "object": {
                    "dense_nodes": 352,
                    "bm25_nodes": 352
               },
               "hyper": {
                    "dense_nodes": 125,
                    "bm25_nodes": 125
               },
               "episodic": {
                    "dense_nodes": 33,
                    "bm25_nodes": 33
               },
               "time": {
                    "dense_nodes": 0,
                    "bm25_nodes": 0
               }
          },
          "triplets": {
               "dense_triplets": 319,
               "bm25_triplets": 319
          }
     },
     "

True

In [21]:
personalai.textid_store.count_items()

{'textid_to_tripletsid': 33, 'tripletid_to_textsid': 1178}

In [22]:
print("mem kv_cache info:")
print(json.dumps(personalai.mem_pipeline.get_cache_stat(), indent=5))
print("mem agent tgen info:")
print(json.dumps(personalai.mem_pipeline.get_agent_tgen_stat(), indent=5))

mem kv_cache info:
{
     "MemPipeline": null,
     "extractor": {
          "LLMExtractor": null,
          "triplets_extraction_solver": 33,
          "thesises_extraction_solver": 33
     },
     "updator": {
          "LLMUpdator": null,
          "replace_simple_solver": 0,
          "replace_hyper_solver": 0
     }
}
mem agent tgen info:
{
     "extractor": {
          "triplets_extraction_solver": {
               "prompt_tokens_amount": {
                    "count": 33,
                    "count_not_null": 33,
                    "min": 912,
                    "max": 1090,
                    "median": 947,
                    "mean": 954.03,
                    "std": 33.506,
                    "sum": 31483
               },
               "generated_tokens_amount": {
                    "count": 33,
                    "count_not_null": 33,
                    "min": 31,
                    "max": 269,
                    "median": 87,
                    "mean": 99.303,


In [23]:
answer1, rinfo1, trace1 = personalai.answer_question("Какими помещениями разрешено пользоваться студентам, проживающим в общежитии МСГ?")
pprint(answer1)

('Студенты, проживающие в общежитии МСГ, имеют доступ к помещениям для '
 'самостоятельных занятий и культурно-бытового назначения. Кроме того, '
 'предусмотрены душевые комплексы с сауной, использование которых регулируется '
 'администрацией ФГБУ «МСГ». Студенты могут участвовать в студенческом совете '
 'и влиять на правила использования этих помещений.')


In [24]:
answer2, rinfo2, trace2 = personalai.answer_question("Разрешено ли студентам, проживающим в общежитии МСГ, пользоваться бытовой техникой?")
pprint(answer2)

('Студенты в общежитии МСГ имеют право пользоваться бытовой техникой, однако '
 'конкретные виды не указаны в правилах проживания. Студенты обязаны соблюдать '
 'правила пожарной и технической безопасности при ее использовании.')


In [25]:
answer3, rinfo3, trace3 = personalai.answer_question("Могут ли проживающие в общежитии МСГ обращаться к администрации с вопросами?")
pprint(answer3)

('Да, проживающие в общежитии МСГ могут обращаться к администрации через '
 'студенческий совет с любыми вопросами.')


In [26]:
answer4, rinfo4, trace4 = personalai.answer_question("Что должны делать люди, проживающие в общежитии МСГ, если они обнаружили неисправный датчик пожарной безопасности?")
pprint(answer4)

('Люди, проживающие в общежитии МСГ, должны немедленно сообщить об неисправном '
 'датчике пожарной безопасности администрации.')


In [27]:
print("qa kv_cache info:")
print(json.dumps(personalai.qa_pipeline.get_cache_stat(), indent=5))
print("qa agent tgen info:")
print(json.dumps(personalai.qa_pipeline.get_agent_tgen_stat(), indent=5))

qa kv_cache info:
{
     "QAPipeline": 4,
     "query_preprocessor": {
          "QueryPreprocessor": 4,
          "denoiser": {
               "QueryDenoiser": 4,
               "swremoval_solver": 4,
               "grammar_check_solver": 4
          },
          "enhancer": null,
          "decomposer": {
               "QueryDecomposer": 4,
               "decompose_classifier_solver": 4,
               "q_decomposition_solver": 1
          }
     },
     "kg_reasoner": {
          "KnowledgeGraphReasoner": null,
          "reasoner": {
               "MediumKGReasoner": 5,
               "searchplan_enhancer": {
                    "SearchPlanEnhancer": 11,
                    "plan_initialing_solver": 5,
                    "enhance_classify_solver": 0,
                    "plan_enhancing_solver": 6,
                    "searchstop_classify_solver": 0
               },
               "entities_extractor": {
                    "EntitiesExtractor": 11,
                    "entitie

In [28]:
personalai.kg_model.count_items()

{'graph_info': {'triplets': 1178, 'nodes': 510},
 'embeddings_info': {'nodes': 510, 'triplets': 319},
 'nodestree_info': None}

In [29]:
personalai.textid_store.count_items()

{'textid_to_tripletsid': 33, 'tripletid_to_textsid': 1178}

In [ ]:
#personalai.mem_pipeline.clear_kv_caches()
#personalai.mem_pipeline.clear_agent_tgen_stat()
personalai.qa_pipeline.clear_kv_caches(clear_traversal_cache=True, clear_retrieval_cache=True)
personalai.qa_pipeline.clear_agent_tgen_stat()

In [ ]:
print("mem kv_cache info:")
print(json.dumps(personalai.mem_pipeline.get_cache_stat(), indent=5))
print("mem agent tgen info:")
print(json.dumps(personalai.mem_pipeline.get_agent_tgen_stat(), indent=5))
print("qa kv_cache info:")
print(json.dumps(personalai.qa_pipeline.get_cache_stat(), indent=5))
print("qa agent tgen info:")
print(json.dumps(personalai.qa_pipeline.get_agent_tgen_stat(), indent=5))

In [ ]:
existing_textid = "96222e0c62218faad5d5e1dd1a04fcdf" # TO CHANGE
delete_info, rinfo, trace = personalai.clear_memory(existing_textid)

In [ ]:
personalai.kg_model.count_items()

In [ ]:
personalai.textid_store.count_items()

In [30]:
personalai.close_connections()
del personalai

In [ ]:
personalai = PersonalAI()
personalai.open_connections()
answer, rinfo, trace = personalai.answer_question("Что должны делать люди, проживающие в общежитии МСГ, если они обнаружили неисправный датчик пожарной безопасности?")
pprint(answer)
personalai.close_connections()

Existing PAI environment will be loaded.


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9271.60it/s]


In [35]:
personalai.close_connections()